## <u>Import Libraries</u>

In [1]:
import gymnasium as gym
import numpy as np
import random
import time

---
## <u>Q-learning: theory, and how it differs from SARSA</u>

Q-learning is an **off-policy** temporal difference control algorithm, in contrast to SARSA's **on-policy** approach.

**Update rule**:
```
Q(s,a) = Q(s,a) + alpha * [R + gamma * max_a' Q(s',a') - Q(s,a)]
```

The only mathematical difference from SARSA's update rule is this: SARSA uses `Q(s',a')`, the value of the action the policy will *actually* take next (chosen via epsilon-greedy, exploration included). Q-learning uses `max_a' Q(s',a')`, the value of the *best possible* action from the next state, regardless of what the policy will actually do next.

**Why action selection moves inside the loop here, unlike SARSA:** SARSA's update genuinely depends on a specific next action, so it has to be computed once before the loop (the first action) and then carried forward, one step ahead, every iteration (this episode's "next action" becomes next iteration's "current action"). Q-learning's update only ever needs `max()` over the next state's whole row of Q-values, no specific action required at all. There is nothing to carry forward between iterations, so the current action is simply chosen fresh, at the top of every loop iteration, based on whatever the current state happens to be at that point.

**Practical consequence of this difference**: Q-learning's target value assumes the agent will always act optimally on the next step, even though epsilon-greedy means it sometimes won't (it might explore randomly instead). This makes Q-learning's learned values a slightly optimistic estimate of what the current policy will actually achieve, since the update doesn't account for the policy's own exploration risk. SARSA's target, by contrast, is grounded in what the policy will actually do next, exploration risk included. This difference is small in the update formula, but it produces a real, visible behavioural difference in the final learned policy, covered in the findings section at the end.

---
## <u>Train</u>

In [2]:
# make Q-values table
Q_values = np.zeros((48,4))

# initialise hyperparameters
gamma = 0.99
alpha = 0.5
epsilon = 0.1
episodes = 500

# define epsilon greedy
def epsilon_greedy(curr_state):

    if random.random() < epsilon:
        return env.action_space.sample()       # explore: random action
    else:
        return np.argmax(Q_values[curr_state])  # exploit: best known action for this state


start_time = time.perf_counter()

for episode in range(episodes):

    # every 50th episode gets a renderable (visual) environment, every other episode a normal one
    if episode % 50 == 0:
        env = gym.make("CliffWalking-v1", render_mode = "human")
    else:
        env = gym.make("CliffWalking-v1")

    curr_state, extra = env.reset()

    total_rewards = 0
    total_steps = 0

    terminated = False
    truncated = False

    while not (terminated or truncated):

        # chosen fresh every iteration, based on the current state, Q-learning doesn't need to
        # carry a "next action" forward between iterations the way SARSA does, see markdown above
        curr_action = epsilon_greedy(curr_state)

        next_state, reward, terminated, truncated, extra = env.step(curr_action)

        # Q-learning update: max over the WHOLE next-state row, not a specific next action
        # this is the one line that differs from SARSA's update, and it's what makes this off-policy
        td_target = reward + gamma * np.max(Q_values[next_state])
        td_error = td_target - Q_values[curr_state, curr_action]
        Q_values[curr_state, curr_action] = Q_values[curr_state, curr_action] + alpha * td_error

        total_rewards = total_rewards + reward
        total_steps = total_steps + 1

        curr_state = next_state

    if episode % 50 == 0:
        print(f"for episode {episode+1}/{episodes} --------> total rewards = {total_rewards} -------> total steps = {total_steps}")  

env.close()

end_time = time.perf_counter()
print(f"Visual Training finished in {end_time - start_time:.2f} seconds.")

for episode 1/500 --------> total rewards = -158 -------> total steps = 158
for episode 51/500 --------> total rewards = -237 -------> total steps = 39
for episode 101/500 --------> total rewards = -13 -------> total steps = 13
for episode 151/500 --------> total rewards = -13 -------> total steps = 13
for episode 201/500 --------> total rewards = -122 -------> total steps = 23
for episode 251/500 --------> total rewards = -15 -------> total steps = 15
for episode 301/500 --------> total rewards = -13 -------> total steps = 13
for episode 351/500 --------> total rewards = -13 -------> total steps = 13
for episode 401/500 --------> total rewards = -13 -------> total steps = 13
for episode 451/500 --------> total rewards = -115 -------> total steps = 16
Visual Training finished in 83.84 seconds.


---
## <u>Test the trained policy</u>

Pure exploitation here, no epsilon-greedy at all: `np.argmax(Q_values[curr_state])` always takes the best known action, since this is evaluating the learned policy, not still learning it.

In [3]:
env = gym.make("CliffWalking-v1", render_mode = "human")

total_rewards = 0
total_steps = 0
terminated = False
truncated = False

curr_state, extra = env.reset()

while not (terminated or truncated):

    curr_action = np.argmax(Q_values[curr_state])  # always exploit, no exploration during testing

    next_state, reward, terminated, truncated, extra = env.step(curr_action)

    curr_state = next_state

    total_rewards = total_rewards + reward
    total_steps = total_steps + 1

env.close()
print(f"total reward = {total_rewards} & episode len = {total_steps}")

total reward = -13 & episode len = 13


---
## <u>Report: SARSA vs Q-learning findings</u>

Q-learning's final tested policy achieved a total reward of **-13** in **13 steps**. This is the mathematically optimal outcome on this grid: the Manhattan distance from the start cell to the goal cell is exactly 13 steps (11 across, 1 up, 1 down), so no policy can do better than -13 (one step of -1 reward each, no cliff penalty). Q-learning found the shortest possible route, which runs directly along the edge of the cliff, the riskiest path available.

SARSA's tested policy, by contrast, settled on a total reward around **-17**, a slightly longer but noticeably safer route, staying a row or two back from the cliff edge rather than hugging it.

**Why this happens, tied directly to the theory above**: Q-learning's update always targets the best possible next action, so it never "prices in" the real risk that its own epsilon-greedy exploration could randomly step off the cliff while walking the edge. It learns the policy that is optimal in a world with no exploration risk, then that policy gets executed with exploration risk still present during training, occasionally paying for it with a cliff fall, but the learned Q-values themselves stay optimistic and cliff-hugging. SARSA's update targets the action its own policy will actually take next, exploration included, so it learns that the cliff-edge path is more dangerous than it looks once real exploration noise is accounted for, and settles on the safer route instead.

This is the same result Sutton and Barto's original Cliff Walking example is built to demonstrate: Q-learning converges to the optimal but risky path, SARSA converges to a safer, slightly suboptimal path, precisely because one algorithm learns off-policy and the other on-policy. Seeing it reproduced here, from a from-scratch implementation rather than a textbook figure, is a genuinely solid confirmation that both algorithms were implemented correctly.